In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from tqdm import tqdm

# ================= 配置区 =================
STOCK_DATA_PATH = r"D:\work\trade\data_center\storage\market_data\stock_daily"
OUTPUT_PATH = r"D:\work\data2\train_data.csv"

# =====================================================
# =============== Part 1：生成大盘情绪数据 ===============
# =====================================================

def process_stock_data(folder_path):
    file_list = glob.glob(os.path.join(folder_path, "*.parquet"))
    if not file_list:
        return None

    processed_dfs = []

    for file_path in tqdm(file_list, desc="Building Market Data"):
        try:
            df = pd.read_parquet(file_path)
            if df.empty:
                continue

            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)

            df = df.sort_index()
            df = df.dropna(subset=['close'])
            if df.empty:
                continue

            df['pct_chg'] = df['close'].pct_change()
            eps = 1e-4
            df['is_limit_up'] = (df['close'] >= df['limit_up'] - eps).astype(int)
            df['is_limit_down'] = (df['close'] <= df['limit_down'] + eps).astype(int)
            df['is_valid'] = 1

            processed_dfs.append(
                df[['is_limit_up', 'is_limit_down', 'pct_chg', 'is_valid']]
            )

        except Exception:
            continue

    if not processed_dfs:
        return None

    all_stocks = pd.concat(processed_dfs)

    daily = all_stocks.groupby(all_stocks.index).agg({
        'is_limit_up': 'sum',
        'is_limit_down': 'sum',
        'is_valid': 'sum',
        'pct_chg': 'mean'
    })

    daily['limit_up_ratio'] = daily['is_limit_up'] / daily['is_valid']
    daily['limit_down_ratio'] = daily['is_limit_down'] / daily['is_valid']

    final_df = daily.rename(columns={
        'is_limit_up': 'limit_up_count',
        'is_limit_down': 'limit_down_count',
        'pct_chg': 'avg_pct_change'
    })

    final_df = final_df[
        ['limit_up_count', 'limit_down_count',
         'limit_up_ratio', 'limit_down_ratio',
         'avg_pct_change']
    ]

    final_df = final_df.reset_index().rename(columns={'index': 'date'})
    final_df['date'] = pd.to_datetime(final_df['date'])
    final_df.set_index('date', inplace=True)

    return final_df


# =====================================================
# =============== Part 2：个股特征 & 标签 ===============
# =====================================================

def calculate_technical_features(df):
    df['ma5'] = df['close'].rolling(5).mean()
    df['ma_vol_5'] = df['volume'].rolling(5).mean()
    df['pre_close'] = df['close'].shift(1)

    df['upper_shadow'] = (
        df['high'] - df[['open', 'close']].max(axis=1)
    ) / df['close']

    df['body_size'] = abs(df['close'] - df['open']) / df['open']
    df['ma5_bias'] = (df['close'] - df['ma5']) / df['ma5']
    df['vol_ratio'] = df['volume'] / df['ma_vol_5'].replace(0, np.nan)
    df['high_low_ratio'] = (df['high'] - df['low']) / df['pre_close']

    # ========= 标签 =========
    next_close = df['close'].shift(-1)
    next_open = df['open'].shift(-1)
    next_high = df['high'].shift(-1)

    cond_1 = next_close > next_open
    cond_2 = (next_high - next_open) / next_open > 0.03
    df['label'] = (cond_1 & cond_2).astype(int)

    # ========= 新增：第二天开盘涨幅 =========
    df['next_open_return'] = next_open / df['close'] - 1

    return df


def apply_strategy_filter(df):
    df['is_limit_up'] = (df['close'] >= df['limit_up'] - 1e-4).astype(int)
    df['recent_limit_ups_20'] = df['is_limit_up'].shift(1).rolling(60).sum()

    df['L1'] = df['is_limit_up'].shift(1) == 1
    df['L2'] = df['is_limit_up'].shift(2) == 1
    df['L3'] = df['is_limit_up'].shift(3) == 1

    df['open_T1'] = df['open'].shift(1)
    df['open_T2'] = df['open'].shift(2)
    df['open_T3'] = df['open'].shift(3)

    not_overheated = df['recent_limit_ups_20'] < 6

    past_1 = df['is_limit_up'].shift(2).rolling(6).sum()
    case_1 = (
        df['L1'] & (~df['is_limit_up'].astype(bool)) &
        (past_1 < 2) & (df['close'] >= df['open_T1'])
    )

    past_2 = df['is_limit_up'].shift(3).rolling(6).sum()
    case_2 = (
        df['L2'] & (~df['L1']) & (~df['is_limit_up'].astype(bool)) &
        (past_2 < 2) & (df['close'] >= df['open_T2'])
    )

    past_3 = df['is_limit_up'].shift(4).rolling(6).sum()
    case_3 = (
        df['L3'] & (~df['L2']) & (~df['L1']) &
        (~df['is_limit_up'].astype(bool)) &
        (past_3 < 2) & (df['close'] >= df['open_T3'])
    )

    mask = (case_1 | case_2 | case_3) & not_overheated & (df['is_valid'] == 1)
    return df[mask].copy()


# =====================================================
# ======================= 主流程 ========================
# =====================================================

def main():
    df_market = process_stock_data(STOCK_DATA_PATH)
    if df_market is None or df_market.empty:
        return

    market_cols = df_market.columns.tolist()
    all_files = glob.glob(os.path.join(STOCK_DATA_PATH, "*.parquet"))
    pattern = re.compile(r'^(00|60)\d+')

    valid_files = [
        f for f in all_files
        if pattern.match(os.path.basename(f).split('.')[0])
    ]

    result_list = []

    for file_path in tqdm(valid_files, desc="Processing Stocks"):
        try:
            df = pd.read_parquet(file_path)
            if df.empty:
                continue

            if not isinstance(df.index, pd.DatetimeIndex):
                df.index = pd.to_datetime(df.index)

            df = df.sort_index()
            df = df.dropna(subset=['close'])
            df['is_valid'] = 1

            df = calculate_technical_features(df)
            selected_df = apply_strategy_filter(df)
            if selected_df.empty:
                continue

            combined = selected_df.join(df_market, how='left')

            # ★ 关键：强制把 index 变成 date 列
            combined = combined.reset_index().rename(columns={'index': 'date'})

            combined['code'] = os.path.basename(file_path).split('.')[0]


            cols = [
                'date', 'code',
                'upper_shadow', 'body_size', 'ma5_bias',
                'vol_ratio', 'high_low_ratio',
            ] + market_cols + [
                'label', 'next_open_return'
            ]

            result_list.append(combined[cols])

        except Exception:
            continue

    if not result_list:
        return

    train_data = pd.concat(result_list)

    train_data = train_data.sort_index().sort_values('code')


    # 同时清理 label 和 next_open_return 的 NaN
    train_data = train_data.dropna(
        subset=['label', 'next_open_return']
    )

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    train_data.to_csv(OUTPUT_PATH, index=False)

    print(f"✅ 完成，样本数: {len(train_data)}")
    print(f"📁 保存至: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Building Market Data: 100%|██████████| 5176/5176 [01:14<00:00, 69.26it/s]


KeyError: 'date'